# POMDP Candidate Selector

Notebook-first inspection of the first belief-limited attacker selector.

This is not a trained policy yet. It ranks VAE candidates using noisy attacker observations and profile/intent metadata, then compares that ranking to the full-state Monte Carlo selector when ranked-candidate artifacts are available.


## Imports


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve repo root when this notebook is launched from either repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from convoy_sim.feasibility import Environment
from convoy_sim.pomdp_candidate_selector import (
    build_candidate_observation_rows,
    rank_candidate_observation_rows,
    write_belief_ranked_csv,
    write_belief_ranked_json,
)
from convoy_sim.profile_generation_viz import FIXED_X_LIMITS, apply_limits, combined_limits, simple_ship_polygons, style_ax
from convoy_sim.realism import get_attacker_observation_config
from experiments.evaluate_attack_candidate_pool import load_candidate_records
from scenarios.convoy_profiles import get_convoy_layout_profile

PROJECT_ROOT


## Config


In [ ]:
CANDIDATE_PATH = Path("data/attack_profiles/vae_candidates/mixed_curated70_random30_hit_candidates.jsonl")
FULL_STATE_RANKED_CSV = Path("results/runs/candidate_pool_eval/20260511_211859_vae_source_compare_mixed_70_30/ranked_candidates.csv")
CONVOY_PROFILE = "convoy_layout_1"
MAX_PROFILES = 100
TOP_K = 25
SEED = 1945

OBSERVATION_PRESET = "good_contact"
OBS_CFG = get_attacker_observation_config(OBSERVATION_PRESET)
ENV = Environment(time_of_day="night", visibility_m=3500.0, sea_state=4)

OUTPUT_DIR = PROJECT_ROOT / "results" / "diag" / "pomdp_candidate_selector"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Build Belief-Limited Ranking


In [ ]:
records_all = load_candidate_records(PROJECT_ROOT / CANDIDATE_PATH)
records = records_all[: int(MAX_PROFILES)] if MAX_PROFILES is not None else records_all
ships = get_convoy_layout_profile(CONVOY_PROFILE).build_ships()

observation_rows = build_candidate_observation_rows(
    records,
    ships=ships,
    seed=int(SEED),
    env=ENV,
    observation_cfg=OBS_CFG,
)
ranked_rows = rank_candidate_observation_rows(observation_rows)

belief_df = pd.DataFrame(ranked_rows)
write_belief_ranked_csv(OUTPUT_DIR / "belief_ranked_candidates.csv", ranked_rows)
write_belief_ranked_json(OUTPUT_DIR / "top_belief_candidates.json", ranked_rows[: int(TOP_K)])
belief_df.head(10)


## Summary


In [ ]:
summary = {
    "candidate_path": str(PROJECT_ROOT / CANDIDATE_PATH),
    "profiles_ranked": int(len(belief_df)),
    "top_k": int(TOP_K),
    "best_profile_id": str(belief_df.iloc[0]["profile_id"]) if len(belief_df) else "",
    "best_belief_score": float(belief_df.iloc[0]["belief_score"]) if len(belief_df) else np.nan,
    "mean_belief_score": float(belief_df["belief_score"].mean()) if len(belief_df) else np.nan,
    "observation_preset": OBSERVATION_PRESET,
    "observation_config": OBS_CFG.to_dict(),
}
(OUTPUT_DIR / "belief_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
summary


## Belief Feature Diagnostics


In [ ]:
display_cols = [
    "belief_rank",
    "profile_id",
    "belief_score",
    "estimated_range_m",
    "bearing_alignment_score",
    "aspect_score",
    "spread_score",
    "inside_score",
    "uncertainty_score",
    "spawn_region",
    "approach_side",
]
belief_df[display_cols].head(int(TOP_K))


In [ ]:
component_cols = ["range_score", "bearing_alignment_score", "aspect_score", "spread_score", "contact_score", "inside_score", "uncertainty_score"]
component_summary = belief_df[component_cols].describe().T
component_summary.to_csv(OUTPUT_DIR / "belief_component_summary.csv")
component_summary


## Compare To Full-State Ranking


In [ ]:
if (PROJECT_ROOT / FULL_STATE_RANKED_CSV).exists():
    full_state_df = pd.read_csv(PROJECT_ROOT / FULL_STATE_RANKED_CSV)
    compare_df = belief_df.merge(
        full_state_df[["profile_id", "rank", "attacker_score", "expected_loss", "expected_hits", "expected_unique_ships_hit", "value_lost"]],
        on="profile_id",
        how="left",
        suffixes=("_belief", "_full_state"),
    )
    compare_df = compare_df.rename(columns={"rank": "full_state_rank"})
    compare_df["in_full_state_top_k"] = compare_df["full_state_rank"].le(int(TOP_K))
    compare_df["in_belief_top_k"] = compare_df["belief_rank"].le(int(TOP_K))
    overlap = int((compare_df["in_full_state_top_k"] & compare_df["in_belief_top_k"]).sum())
    comparison_summary = {
        "belief_top_k": int(TOP_K),
        "full_state_top_k": int(TOP_K),
        "top_k_overlap": overlap,
        "top_k_overlap_rate": float(overlap / max(int(TOP_K), 1)),
        "belief_best_profile_id": str(compare_df.sort_values("belief_rank").iloc[0]["profile_id"]),
        "full_state_best_profile_id": str(full_state_df.sort_values("rank").iloc[0]["profile_id"]),
    }
    compare_df.to_csv(OUTPUT_DIR / "belief_vs_full_state_ranked.csv", index=False)
    (OUTPUT_DIR / "belief_vs_full_state_summary.json").write_text(json.dumps(comparison_summary, indent=2) + "\n", encoding="utf-8")
else:
    full_state_df = pd.DataFrame()
    compare_df = belief_df.copy()
    comparison_summary = {"warning": f"Missing full-state ranked CSV: {PROJECT_ROOT / FULL_STATE_RANKED_CSV}"}
comparison_summary


In [ ]:
if "full_state_rank" in compare_df.columns:
    compare_df[[
        "belief_rank",
        "full_state_rank",
        "profile_id",
        "belief_score",
        "expected_loss",
        "expected_hits",
        "expected_unique_ships_hit",
        "spawn_region",
    ]].sort_values("belief_rank").head(int(TOP_K))


## Plots


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor="white")

belief_df["belief_score"].plot.hist(ax=axes[0], bins=30, color="#1b9e77", edgecolor="white")
axes[0].set_title("Belief score distribution")
axes[0].set_xlabel("belief score")

belief_df["estimated_range_m"].plot.hist(ax=axes[1], bins=30, color="#377eb8", edgecolor="white")
axes[1].set_title("Noisy estimated range")
axes[1].set_xlabel("m")

belief_df["spawn_region"].value_counts().plot.bar(ax=axes[2], color="#4daf4a")
axes[2].set_title("Candidate regions")
axes[2].set_xlabel("")
axes[2].set_ylabel("count")

for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
if "full_state_rank" in compare_df.columns:
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    plot_df = compare_df.dropna(subset=["full_state_rank"])
    ax.scatter(plot_df["belief_rank"], plot_df["full_state_rank"], s=28, alpha=0.7, color="#111111")
    ax.axline((1, 1), slope=1, color="#d95f02", linestyle="--", linewidth=1.2)
    ax.set_title("Belief rank vs full-state rank")
    ax.set_xlabel("belief-limited rank")
    ax.set_ylabel("full-state rank")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
ship_xy = np.asarray([ship.position for ship in ships], dtype=float)
candidate_xy = belief_df[["u_pos_x", "u_pos_y"]].to_numpy(dtype=float) if {"u_pos_x", "u_pos_y"} <= set(belief_df.columns) else None
if candidate_xy is None:
    candidate_xy = np.asarray([record["profile"]["u_pos"] for record in records], dtype=float)
    belief_df["u_pos_x"] = candidate_xy[:, 0]
    belief_df["u_pos_y"] = candidate_xy[:, 1]
limits = combined_limits(ships, extra_points=candidate_xy, pad=350.0)

top_belief = belief_df.head(int(TOP_K)).copy()
fig, ax = plt.subplots(figsize=(10, 8), facecolor="lightgrey")
style_ax(ax, "Belief-limited top candidates")
simple_ship_polygons(ax, ships)
ax.scatter(candidate_xy[:, 0], candidate_xy[:, 1], c="#111111", s=10, alpha=0.16, label="candidate pool", zorder=2)
ax.scatter(top_belief["u_pos_x"], top_belief["u_pos_y"], c="#d95f02", s=28, alpha=0.85, label="belief top-k", zorder=3)
apply_limits(ax, limits, fixed_x_limits=FIXED_X_LIMITS)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend(frameon=True, facecolor="lightgrey", edgecolor="black", framealpha=1.0)
plt.tight_layout()
plt.show()


## Artifact Paths


In [ ]:
{
    "belief_ranked_candidates_csv": str(OUTPUT_DIR / "belief_ranked_candidates.csv"),
    "top_belief_candidates_json": str(OUTPUT_DIR / "top_belief_candidates.json"),
    "belief_summary_json": str(OUTPUT_DIR / "belief_summary.json"),
    "belief_component_summary_csv": str(OUTPUT_DIR / "belief_component_summary.csv"),
    "belief_vs_full_state_ranked_csv": str(OUTPUT_DIR / "belief_vs_full_state_ranked.csv"),
    "belief_vs_full_state_summary_json": str(OUTPUT_DIR / "belief_vs_full_state_summary.json"),
}
